# 01. Data Cleaning & Integration

This notebook performs the fundamental data cleaning operations on internal Hungry Hub data (transactions, reviews, tags, KOLs, and web analytics), producing a master dataset ready for web scraping deduplication.

## Key Fixes Implemented Here:
- **Revenue Inflation**: Groups transaction data (`dim_v2.csv`) by identical booking items first before aggregation, as the raw data contains duplicated rows per booking item/payment method.
- **Web Analytics Coverage**: Replaces brittle regex URL parsing with an intelligent lookup to assign `web_views` by matching actual slugs from the `Restaurant DB All` reference table.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Setup Paths

In [2]:
# Relative to the Feature_Engineering_Ready folder
DATA_DIR = "Data Sources"
OUTPUT_DIR = "Output_Data"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## 2. Load and Clean Transactions (`dim_v2.csv`)
The core issue previously was massive revenue inflation. We will deduplicate transactions by `id` (booking ID) before summing up revenue, as transactions are duplicated for different item rows or payment methods in the raw SQL dump.

In [3]:
print("Loading Transactions...")
df_trans = pd.read_csv(os.path.join(DATA_DIR, "dim_v2.csv"), low_memory=False)

# Apply Global Exclusions explicitly defined by Operations
df_trans = df_trans[~df_trans['restaurant_id'].isin([4503, 4502, 933, 837])]

# Ensure revenue is numeric 
df_trans['revenue'] = pd.to_numeric(df_trans['revenue'], errors='coerce').fillna(0)

# DEDUPLICATE BY BOOKING ID FIRST TO PREVENT REVENUE OVERCOUNTING
df_trans_unique = df_trans.drop_duplicates(subset=['id'], keep='first').copy()

df_trans_unique['total_guests'] = df_trans_unique['adult'] + df_trans_unique['kids']

# Aggregations per restaurant
agg_funcs = {
    'revenue': 'sum',
    'id': 'count', # Total unique bookings
    'total_guests': 'sum',
    'no_show': 'sum',
    'is_temporary': 'sum'
}

trans_stats = df_trans_unique.groupby('restaurant_id').agg(agg_funcs).reset_index()
trans_stats.rename(columns={
    'revenue': 'total_revenue',
    'id': 'total_bookings',
    'no_show': 'total_no_shows',
    'is_temporary': 'total_temporary_bookings'
}, inplace=True)

trans_stats['avg_revenue_per_booking'] = trans_stats['total_revenue'] / trans_stats['total_bookings'].replace(0, np.nan)
trans_stats['no_show_rate'] = trans_stats['total_no_shows'] / trans_stats['total_bookings'].replace(0, np.nan)

print(f"Processed {len(trans_stats)} restaurants for transactions. Total Revenue: {trans_stats['total_revenue'].sum():,.2f}")

Loading Transactions...


Processed 3643 restaurants for transactions. Total Revenue: 73,198,212,074.58


## 3. Load and Clean Metadata & Ratings

In [4]:
print("Loading Metadata...")

tags_df = pd.read_csv(os.path.join(DATA_DIR, "dim_tags.csv"))
tags_df = tags_df.drop_duplicates(subset=['restaurant_id']).fillna("Unknown")

bookings_df = pd.read_csv(os.path.join(DATA_DIR, "dim_bookings.csv"))
bookings_df = bookings_df[['restaurant_id', 'name', 'days_in_advance']]

ratings_df = pd.read_csv(os.path.join(DATA_DIR, "dim_customer_rating.csv"))

meta_df = pd.merge(bookings_df, tags_df, on='restaurant_id', how='left')
meta_df = pd.merge(meta_df, ratings_df, on='restaurant_id', how='left')
print(f"Processed {len(meta_df)} metadata records.")

Loading Metadata...
Processed 2474 metadata records.


## 4. Load KOL Influencer Data
Clean the influencer views and engagements to find total KOL reach.

In [5]:
print("Loading KOL Data...")
posts = pd.read_csv(os.path.join(DATA_DIR, "KOL DATA - SMU - Posts.csv"))

def clean_num(x):
    if pd.isna(x): return 0
    if isinstance(x, (int, float)): return x
    x = str(x).replace(',', '')
    if 'k' in x.lower():
        try: return float(x.lower().replace('k', '')) * 1000
        except: pass
    try: return float(x)
    except: return 0

for col in ['Views', 'Likes', 'Comments']:
    posts[col] = posts[col].apply(clean_num)

posts['restaurant_id'] = pd.to_numeric(posts['Restaurant Code'], errors='coerce')

kol_stats = posts.groupby('restaurant_id')[['Views', 'Likes', 'Comments']].sum().reset_index()
kol_stats.columns = ['restaurant_id', 'kol_views', 'kol_likes', 'kol_comments']
kol_count = posts.groupby('restaurant_id')['KOL_ID'].nunique().reset_index(name='unique_kols_used')

kol_df = pd.merge(kol_stats, kol_count, on='restaurant_id', how='left')
print(f"Processed {len(kol_df)} KOL records.")

Loading KOL Data...


Processed 235 KOL records.


## 5. Web Analytics (Fixing missing mapping)
Instead of exact regex extraction, we use `slugs` from the restaurant database reference table to map GA `pagePath` hits correctly.

In [6]:
print("Loading Web Analytics...")
analytics = pd.read_csv(os.path.join(DATA_DIR, "analytics_data (1).csv"))

# Load Restaurant DB for links to match page paths
r_db = pd.read_csv(os.path.join(DATA_DIR, "KOL DATA - SMU - Restaurant DB All.csv"))

def extract_slug(url):
    if pd.isna(url): return None
    return str(url).strip().split('/')[-1].split('?')[0]

r_db['slug'] = r_db['link'].apply(extract_slug)
slug_map = r_db.set_index('slug')['ID'].to_dict()

# Apply logic to map analytics pagePath via slug mapping
def extract_path_slug(path):
    if pd.isna(path): return None
    if 'restaurants/' in str(path):
        parts = path.split('restaurants/')
        if len(parts) > 1:
            return parts[1].split('/')[0].split('?')[0]
    return None

analytics['slug'] = analytics['pagePath'].apply(extract_path_slug)
analytics['restaurant_id'] = analytics['slug'].map(slug_map)

matched = analytics.dropna(subset=['restaurant_id']).copy()
analytics_df = matched.groupby('restaurant_id').agg({
    'screenPageViews': 'sum',
    'purchaseRevenue': 'sum'
}).reset_index().rename(columns={
    'screenPageViews': 'web_views',
    'purchaseRevenue': 'web_revenue'
})
print(f"Processed {len(analytics_df)} Web Analytics records.")

Loading Web Analytics...

Processed 872 Web Analytics records.


## 6. Master Integration

In [7]:
# Consolidate all unique IDs
all_ids = set()
for df in [trans_stats, meta_df, kol_df, analytics_df]:
    if 'restaurant_id' in df.columns:
        all_ids.update(df['restaurant_id'].dropna().unique())

master = pd.DataFrame({'restaurant_id': list(all_ids)})

# Join everything
master = pd.merge(master, meta_df, on='restaurant_id', how='left')
master = pd.merge(master, trans_stats, on='restaurant_id', how='left')
master = pd.merge(master, kol_df, on='restaurant_id', how='left')
master = pd.merge(master, analytics_df, on='restaurant_id', how='left')

# ---------------------------------------------------------------------------------------
# PREPARE REQUIRED COLUMNS FOR WEB SCRAPING 
# The Playwright scraping stage heavily relies on `restaurant_name_en`, `restaurant_name_th`
# and an `extracted_location` to reliably search for the restaurants.
# We will extract the exact fields Google Maps, Trip Advisor, etc. require here.
# ---------------------------------------------------------------------------------------
r_db_sub = r_db[['ID', 'Name-en', 'Name-th', 'Primary-Location']].copy()
r_db_sub.rename(columns={'ID': 'restaurant_id'}, inplace=True)

master = pd.merge(master, r_db_sub, on='restaurant_id', how='left')

# Fill name columns using r_db if available, else 'name'
master['restaurant_name_en'] = master['Name-en'].fillna(master['name'])
master['restaurant_name_th'] = master['Name-th']

def extract_location(row):
    # Try DB Primary Location
    loc = row.get('Primary-Location')
    if pd.notna(loc) and str(loc).strip() != '':
        return loc
    # Try Tag Places
    places = row.get('places')
    if pd.notna(places) and str(places).strip() != 'Unknown':
        for t in places.split(','):
            if 'popularzone:' in t:
                return t.split(':')[1].strip().title()
    return 'Bangkok' # Fallback default

master['extracted_location'] = master.apply(extract_location, axis=1)

master.drop(columns=['Name-en', 'Name-th', 'Primary-Location'], inplace=True)

# Null fill purely numeric metrics to 0
metric_cols = [
    'total_revenue', 'total_bookings', 'total_guests', 
    'total_no_shows', 'total_temporary_bookings',
    'kol_views', 'kol_likes', 'kol_comments', 'unique_kols_used',
    'web_views', 'web_revenue'
]
for col in metric_cols:
    master[col] = master[col].fillna(0)

# Add robust KOL Engagement Rate calculation replacing zero-division issue
master['kol_engagement_rate'] = (master['kol_likes'] + master['kol_comments']) / np.maximum(master['kol_views'], 1)

out_file = os.path.join(OUTPUT_DIR, "master_cleaned_data.csv")
master = master[~master['restaurant_id'].isin([4503, 4502, 933, 837])]
master.to_csv(out_file, index=False)
print(f"\nSUCCESS! Saved cleaned master to: {out_file}")
print(f"Final Master Dimensions: {master.shape}")
master.head()


SUCCESS! Saved cleaned master to: Output_Data\master_cleaned_data.csv
Final Master Dimensions: (3727, 29)


,restaurant_id,name,days_in_advance,primary_cuisine,secondary_cuisines,primary_dining_style,secondary_dining_styles,facilities,primary_place,places,...,kol_views,kol_likes,kol_comments,unique_kols_used,web_views,web_revenue,restaurant_name_en,restaurant_name_th,extracted_location,kol_engagement_rate
0,33.0,Audrey Cafe Thonglor Soi 11,90.0,fusion,"italian, international, thai, fusion, european",casual dining,"family friendly, cafe/dessert, chain restauran...","private room up to 10 people, have parking, in...",awardbadge:red table award,popularzone:thonglor,...,3552.0,60.0,4.0,1.0,625.0,0.0,Audrey Cafe Thonglor Soi 11,ออเดรย์ คาเฟ่ ทองหล่อ ซ.11,Thai Isaan,0.018018
1,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Rang Mahal Rooftop at Rembrandt Hotel,ราง มาฮาล รูฟท็อป,Indian,0.000000
2,168.0,Attico Cucina Italiana Radisson Blu Plaza Bangkok,60.0,italian,Unknown,hotel restaurant,"family friendly, open air/outdoor, casual dini...","private room up to 10 people, accommodates 50+...",popularzone:asok,popularzone:asok,...,0.0,0.0,0.0,0.0,12.0,0.0,Attico Cucina Italiana Radisson Blu Plaza Bangkok,แอทติโก้ โรงแรม เรดิสัน บลู พลาซ่า กรุงเทพฯ,Fusion,0.000000
3,220.0,The Living Room at Sheraton Grande Sukhumvit A...,30.0,international,Unknown,hotel buffet,"family friendly, hotel buffet, hotel restaurant","live music, indoor seating, have parking, have...",btsroute:bts asok,"btsroute:bts asok, popularzone:asok",...,0.0,0.0,0.0,1.0,113.0,0.0,The Living Room at Sheraton Grande Sukhumvit A...,The Living Room at Sheraton Grande Sukhumvit A...,Thai,0.000000
4,222.0,Rain Tree Cafe at The Athenee Hotel,90.0,international,"thai, japanese, european, fusion, internationa...",hotel buffet,"buffet, hotel restaurant, family friendly, cas...","free wifi, have parking, wheelchair access, ac...",awardbadge:red table award,"btsroute:bts chit lom, popularzone:ploenchit, ...",...,0.0,0.0,0.0,0.0,1558.0,0.0,Rain Tree Cafe at The Athenee Hotel,เรน ทรี คาเฟ่ ดิ แอทธินี โฮเทล แบงค็อก,Thai Southern,0.000000
